# future_mean_apply_nb
参考 [indicators/nb.ipynb](../indicators/nb.ipynb)，这个函数计算的未来窗口均值是指
$$future\_M{A_t} = MA\left( {{a_{t + wait}}} \right)$$

参数：
- `close`：价格数据的2维数组，行表示时间，列表示不同资产
- `window`：未来统计窗口的大小（时间点数量）
- `ewm`：是否使用指数加权移动平均
  - True：使用指数加权，近期数据权重更高
  - False：使用简单移动平均，所有数据权重相等
- `wait`：等待期数，用于避免使用当前时间点的数据
  - 1：从下一个时间点开始计算（推荐）
  - 0：包含当前时间点（可能引入偏差）
- `adjust`：指数加权移动平均的权重调整参数
  - True：使用调整后的权重
  - False：使用标准权重

返回值：
    与输入形状相同的2维数组，包含每个时间点的未来均值标签

## 源码
```python
@njit(cache=True)
def future_mean_apply_nb(close: tp.Array2d,
                         window: int,
                         ewm: bool,
                         wait: int = 1,
                         adjust: bool = False) -> tp.Array2d:

    if ewm:
        out = generic_nb.ewm_mean_nb(close[::-1], window, minp=window, adjust=adjust)[::-1]
    else:
        out = generic_nb.rolling_mean_nb(close[::-1], window, minp=window)[::-1]
    if wait > 0:
        return generic_nb.bshift_nb(out, wait)
    return out
```

## 例子

In [ ]:
import numpy as np
from vectorbt.labels.nb import future_mean_apply_nb

# 示例价格数据
prices = np.array([
    [100],
    [102],
    [101],
    [105],
    [110],
    [108],
    [107],
    [111],
    [115],
    [113]
], dtype=np.float64)

# 计算未来3天的简单均值
future_mean = future_mean_apply_nb(prices, window=3, ewm=False, wait=1)
print("未来3天简单均值：\n", future_mean.flatten())

# fixed_labels_apply_nb
生成固定时间间隔的的价格变化百分比标签：
$$\frac{{未来价格 - 当前价格}}{当前价格}$$
参数
- `close`：价格数据的2维数组，行表示时间，列表示不同资产
- `n`：未来的时间步数
  - 1：下一个时间点（如明日收盘价）
  - 5：第 5 个时间点（如一周后收盘价）
  - 20：第 20 个时间点（如一个月后收盘价）

返回值：与输入形状相同的2维数组，包含价格变化百分比标签
- 正值：价格上涨
- 负值：价格下跌

```python
@njit(cache=True)
def fixed_labels_apply_nb(close: tp.Array2d, n: int) -> tp.Array2d:

    return (generic_nb.bshift_nb(close, n) - close) / close
```

## 例子

In [ ]:
import numpy as np
from vectorbt.labels.nb import fixed_labels_apply_nb

# 示例价格数据
prices = np.array([
    [100],
    [102],
    [101],
    [105],
    [110],
    [108],
    [107],
    [111],
    [115],
    [113]
], dtype=np.float64)

labels = fixed_labels_apply_nb(prices, n=1)
print(labels)

# local_extrema_apply_nb
判断局部极值点：对于一个序列 ${p_1},{p_2}, \cdots ,{p_N}$
- $t$ 为局部极大值当前仅当：${p_t} \ge {p_{t - 1}}\left( {1 + pos\_th} \right),{p_t} \ge {p_{t + 1}}\left( {1 + pos\_th} \right)$
- $t$ 为局部极小值当前仅当：${p_t} \le {p_{t - 1}}\left( {1 - neg\_th} \right),{p_t} \le {p_{t + 1}}\left( {1 - neg\_th} \right)$
- $t=1或N$ 时忽略不存在的一边

参数
- `close`：价格数据的2维数组，行表示时间，列表示不同资产
- `pos_th`：正向（上涨）阈值，用于识别价格上涨的极值点
  - 范围：(0, 1]，例如0.02表示2%
  - 支持标量、1维数组或2维数组
- `neg_th`：负向（下跌）阈值，用于识别价格下跌的极值点
  - 范围：(0, 1]，例如0.02表示2%
  - 支持标量、1维数组或2维数组
- `flex_2d`：是否使用2维灵活索引，支持不同位置使用不同阈值

返回值：与输入形状相同的2维数组，包含局部极值标记
- 1：局部最高点
- -1：局部最低点
- 0: 非极值点

## 源码
```python
@njit(cache=True)
def local_extrema_apply_nb(close: tp.Array2d,
                           pos_th: tp.MaybeArray[float],
                           neg_th: tp.MaybeArray[float],
                           flex_2d: bool = True) -> tp.Array2d:

    pos_th = np.asarray(pos_th)
    neg_th = np.asarray(neg_th)
    out = np.full(close.shape, 0, dtype=np.int64)

    for col in range(close.shape[1]):
        prev_i = 0
        direction = 0

        for i in range(1, close.shape[0]):
            _pos_th = abs(flex_select_auto_nb(pos_th, prev_i, col, flex_2d))
            _neg_th = abs(flex_select_auto_nb(neg_th, prev_i, col, flex_2d))
            if _pos_th == 0:
                raise ValueError("Positive threshold cannot be 0")
            if _neg_th == 0:
                raise ValueError("Negative threshold cannot be 0")

            if direction == 1:
                # Find next high while updating current lows
                if close[i, col] < close[prev_i, col]:
                    prev_i = i
                elif close[i, col] >= close[prev_i, col] * (1 + _pos_th):
                    out[prev_i, col] = -1
                    prev_i = i
                    direction = -1
            elif direction == -1:
                # Find next low while updating current highs
                if close[i, col] > close[prev_i, col]:
                    prev_i = i
                elif close[i, col] <= close[prev_i, col] * (1 - _neg_th):
                    out[prev_i, col] = 1
                    prev_i = i
                    direction = 1
            else:
                # Find first high/low
                if close[i, col] >= close[prev_i, col] * (1 + _pos_th):
                    out[prev_i, col] = -1
                    prev_i = i
                    direction = -1
                elif close[i, col] <= close[prev_i, col] * (1 - _neg_th):
                    out[prev_i, col] = 1
                    prev_i = i
                    direction = 1

            if i == close.shape[0] - 1:
                # Find last high/low
                if direction != 0:
                    out[prev_i, col] = -direction
    return out
```

## 例子

In [ ]:
import numpy as np
from vectorbt.labels.nb import local_extrema_apply_nb

# 示例价格数据
prices = np.array([
    [100],
    [102],
    [101],
    [105],
    [110],
    [108],
    [107],
    [111],
    [115],
    [113]
], dtype=np.float64)

result1 = local_extrema_apply_nb(prices, pos_th=0.002, neg_th=0.002)
print(result1)
result2 = local_extrema_apply_nb(prices, pos_th=0.02, neg_th=0.02)
print(result2)

# bn_trend_labels_nb
根据已识别的局部极值点（波峰和波谷），为每个区间分配 *上涨* 或 *下跌* 标签。参考 [labels/enums.ipynb](./enums.ipynb) 中的 `Binary` 即二进制趋势标签模式。

参数
- `close`：价格数据的 2 维数组
- `local_extrema`：局部极值点数组，由 `local_extrema_apply_nb` 函数生成
  - 1：波峰点
  - -1：波谷点
  - 0：非极值点

返回：与输入形状相同的2维数组，包含二进制趋势标签
  - 0：下跌趋势（从波峰到波谷）
  - 1：上涨趋势（从波谷到波峰）
  - NaN：极值点之外的区域

## 源码
```python
@njit(cache=True)
def bn_trend_labels_nb(close: tp.Array2d, local_extrema: tp.Array2d) -> tp.Array2d:

    out = np.full_like(close, np.nan, dtype=np.float64)

    for col in range(close.shape[1]):
        idxs = np.flatnonzero(local_extrema[:, col])
        if idxs.shape[0] == 0:
            continue

        for k in range(1, idxs.shape[0]):
            prev_i = idxs[k - 1]
            next_i = idxs[k]

            if close[next_i, col] > close[prev_i, col]:
                out[prev_i:next_i, col] = 1
            else:
                out[prev_i:next_i, col] = 0

    return out
```

In [ ]:
import numpy as np
from vectorbt.labels.nb import local_extrema_apply_nb, bn_trend_labels_nb

# 示例价格数据
prices = np.array([
    [100],
    [102],
    [101],
    [105],
    [110],
    [108],
    [107],
    [111],
    [115],
    [113]
], dtype=np.float64)

result1 = local_extrema_apply_nb(prices, pos_th=0.002, neg_th=0.002)
print(f"局部极值点\n{result1}")
result = bn_trend_labels_nb(prices, result1)
print(f"趋势标签\n{result}")

# bn_cont_trend_labels_nb
参考 [labels/enums.ipynb](./enums.ipynb) 中的 `BinaryCont` 即连续二进制趋势标签模式。

参数
- `close`：价格数据的2维数组
- `local_extrema`：局部极值点数组，由 `local_extrema_apply_nb` 函数生成

返回：与输入形状相同的2维数组，包含连续趋势标签
- 0：表示将会下跌（当前价格接近区间最高点）
- 1：表示将会上涨（当前价格接近区间最低点）
- 中间值：表示上涨或下跌的程度
- NaN：极值点之外的区域

## 源码
```python
@njit(cache=True)
def bn_cont_trend_labels_nb(close: tp.Array2d, local_extrema: tp.Array2d) -> tp.Array2d:

    out = np.full_like(close, np.nan, dtype=np.float64)

    for col in range(close.shape[1]):
        idxs = np.flatnonzero(local_extrema[:, col])
        if idxs.shape[0] == 0:
            continue

        for k in range(1, idxs.shape[0]):
            prev_i = idxs[k - 1]
            next_i = idxs[k]

            _min = np.min(close[prev_i:next_i + 1, col])
            _max = np.max(close[prev_i:next_i + 1, col])
            out[prev_i:next_i, col] = 1 - (close[prev_i:next_i, col] - _min) / (_max - _min)

    return out
```

## 代码

In [ ]:
import numpy as np
from vectorbt.labels.nb import local_extrema_apply_nb, bn_cont_trend_labels_nb

# 示例价格数据
prices = np.array([
    [100],
    [102],
    [101],
    [105],
    [110],
    [108],
    [107],
    [111],
    [115],
    [113]
], dtype=np.float64)

print(f"价格\n{prices}")
result1 = local_extrema_apply_nb(prices, pos_th=0.002, neg_th=0.002)
print(f"局部极值点\n{result1}")
result = bn_cont_trend_labels_nb(prices, result1)
print(f"趋势标签\n{result}")

# bn_cont_sat_trend_labels_nb
参考 [labels/enums.ipynb](./enums.ipynb) 中的 `BinaryContSat` 即饱和连续趋势标签模式。

参数
- `close`：价格数据的2维数组
- `local_extrema`：局部极值点数组
- `pos_th`：正向阈值，用于判断饱和条件
- `neg_th`：负向阈值，用于判断饱和条件
- `flex_2d`：是否使用2维灵活索引

返回：与输入形状相同的2维数组，包含带饱和的连续趋势标签
- 0：强烈的下跌信号（饱和状态）
- 1：强烈的上涨信号（饱和状态）
- 中间值：线性插值的趋势强度
- NaN：极值点之外的区域

## 源码
```python
@njit(cache=True)
def bn_cont_sat_trend_labels_nb(close: tp.Array2d,
                                local_extrema: tp.Array2d,
                                pos_th: tp.MaybeArray[float],
                                neg_th: tp.MaybeArray[float],
                                flex_2d: bool = True) -> tp.Array2d:

    pos_th = np.asarray(pos_th)
    neg_th = np.asarray(neg_th)
    out = np.full_like(close, np.nan, dtype=np.float64)

    for col in range(close.shape[1]):
        idxs = np.flatnonzero(local_extrema[:, col])
        if idxs.shape[0] == 0:
            continue

        for k in range(1, idxs.shape[0]):
            prev_i = idxs[k - 1]
            next_i = idxs[k]

            _pos_th = abs(flex_select_auto_nb(pos_th, prev_i, col, flex_2d))
            _neg_th = abs(flex_select_auto_nb(neg_th, prev_i, col, flex_2d))
            if _pos_th == 0:
                raise ValueError("Positive threshold cannot be 0")
            if _neg_th == 0:
                raise ValueError("Negative threshold cannot be 0")
            _min = np.min(close[prev_i:next_i + 1, col])
            _max = np.max(close[prev_i:next_i + 1, col])

            for i in range(prev_i, next_i):
                if close[next_i, col] > close[prev_i, col]:
                    _start = _max / (1 + _pos_th)
                    _end = _min * (1 + _pos_th)
                    if _max >= _end and close[i, col] <= _start:
                        out[i, col] = 1
                    else:
                        out[i, col] = 1 - (close[i, col] - _start) / (_max - _start)
                else:
                    _start = _min / (1 - _neg_th)
                    _end = _max * (1 - _neg_th)
                    if _min <= _end and close[i, col] >= _start:
                        out[i, col] = 0
                    else:
                        out[i, col] = 1 - (close[i, col] - _min) / (_start - _min)

    return out
```

## 例子

In [ ]:
import numpy as np
from vectorbt.labels.nb import local_extrema_apply_nb, bn_cont_sat_trend_labels_nb

# 示例价格数据
prices = np.array([
    [100],
    [102],
    [101],
    [105],
    [110],
    [108],
    [107],
    [111],
    [115],
    [113]
], dtype=np.float64)

print(f"价格\n{prices}")
result1 = local_extrema_apply_nb(prices, pos_th=0.002, neg_th=0.002)
print(f"局部极值点\n{result1}")
result = bn_cont_sat_trend_labels_nb(prices, result1, pos_th=0.002, neg_th=0.002)
print(f"趋势标签\n{result}")

# pct_trend_labels_nb
参考 [labels/enums.ipynb](./enums.ipynb) 中的 `PctChange` 即百分比变化趋势标签模式。

参数
- `close`：价格数据的2维数组
- `local_extrema`：局部极值点数组
- `normalize`：是否使用标准化计算
  - `True`：使用未来价格作为分母 $$label = \frac{{next\_extrema\_price - current\_price}}{{next\_extrema\_price}}$$
  - `False`：使用当前价格作为分母 $$label = \frac{{next\_extrema\_price{\rm{ }} - {\rm{ }}current\_price}}{{current\_price}}$$

返回：与输入形状相同的2维数组，包含百分比变化标签
- 正值：价格将上涨的百分比
- 负值：价格将下跌的百分比
- NaN：极值点之外的区域

## 源码

```python
@njit(cache=True)
def pct_trend_labels_nb(close: tp.Array2d, local_extrema: tp.Array2d, normalize: bool) -> tp.Array2d:

    out = np.full_like(close, np.nan, dtype=np.float64)

    for col in range(close.shape[1]):
        idxs = np.flatnonzero(local_extrema[:, col])
        if idxs.shape[0] == 0:
            continue

        for k in range(1, idxs.shape[0]):
            prev_i = idxs[k - 1]
            next_i = idxs[k]

            for i in range(prev_i, next_i):
                if close[next_i, col] > close[prev_i, col] and normalize:
                    out[i, col] = (close[next_i, col] - close[i, col]) / close[next_i, col]
                else:
                    out[i, col] = (close[next_i, col] - close[i, col]) / close[i, col]

    return out
```

## 例子

In [ ]:
import numpy as np
from vectorbt.labels.nb import local_extrema_apply_nb, pct_trend_labels_nb

# 示例价格数据
prices = np.array([
    [100],
    [102],
    [101],
    [105],
    [110],
    [108],
    [107],
    [111],
    [115],
    [113]
], dtype=np.float64)

print(f"价格\n{prices}")
result1 = local_extrema_apply_nb(prices, pos_th=0.002, neg_th=0.002)
print(f"局部极值点\n{result1}")
result = pct_trend_labels_nb(prices, result1, normalize=True)
print(f"趋势标签\n{result}")

# trend_labels_apply_nb

## trend_labels_apply_nb
趋势标签生成的统一接口函数：根据指定的模式生成相应的趋势标签。

参数
- `close`：价格数据的2维数组
- `pos_th`：正向（上涨）阈值
- `neg_th`：负向（下跌）阈值
- `mode`：趋势模式，对应TrendMode枚举值
  - `TrendMode.Binary (0)`：二进制趋势标签
  - `TrendMode.BinaryCont (1)`：连续趋势标签
  - `TrendMode.BinaryContSat (2)`：饱和连续趋势标签
  - `TrendMode.PctChange (3)`：百分比变化标签
  - `TrendMode.PctChangeNorm (4)`：标准化百分比变化标签
- `flex_2d`：是否使用2维灵活索引

返回值：与输入形状相同的2维数组，包含相应模式的趋势标签

```python
@njit(cache=True)
def trend_labels_apply_nb(close: tp.Array2d,
                          pos_th: tp.MaybeArray[float],
                          neg_th: tp.MaybeArray[float],
                          mode: int,
                          flex_2d: bool = True) -> tp.Array2d:

    local_extrema = local_extrema_apply_nb(close, pos_th, neg_th, flex_2d)
    if mode == TrendMode.Binary:
        return bn_trend_labels_nb(close, local_extrema)
    if mode == TrendMode.BinaryCont:
        return bn_cont_trend_labels_nb(close, local_extrema)
    if mode == TrendMode.BinaryContSat:
        return bn_cont_sat_trend_labels_nb(close, local_extrema, pos_th, neg_th, flex_2d)
    if mode == TrendMode.PctChange:
        return pct_trend_labels_nb(close, local_extrema, False)
    if mode == TrendMode.PctChangeNorm:
        return pct_trend_labels_nb(close, local_extrema, True)
    raise ValueError("Trend mode is not recognized")
```

# breakout_labels_nb
生成价格突破标签。

原理
- 考虑价格序列 ${p_1}, \cdots ,{p_T}$
- 对于每个时刻 $t$，在未来窗口 $[t + wait,t + wait + w - 1]$ 内，依次检查每个未来时刻 $j$
  - 上涨突破：如果 ${p_j} \ge {p_t}\left( {1 + pos_{th}} \right)$，则 $L_t=1$，并停止检查
  - 下跌突破：如果 ${p_j} \le {p_t}\left( {1 - neg_{th}} \right)$，则 $L_t=-1$，并停止检查
  - 如果窗口内都没有突破，则 $L_t=0$


参数
- `close`：价格数据的 2 维数组
- `window`：未来观察窗口大小（时间点数量）
- `pos_th`：正向突破阈值
  - 例如 0.05 表示 5% 的上涨突破
  - 支持标量或数组格式
- `neg_th`：负向突破阈值
  - 例如 0.05 表示 5% 的下跌突破
  - 支持标量或数组格式
- `wait`：等待期数，避免使用当前时间点
- `flex_2d`：是否使用 2 维灵活索引

返回：与输入形状相同的 2 维数组，包含突破标签
- 1：正向突破（上涨突破）
- -1：负向突破（下跌突破）
- 0：无突破

## 源码

```python
@njit(cache=True)
def breakout_labels_nb(close: tp.Array2d,
                       window: int,
                       pos_th: tp.MaybeArray[float],
                       neg_th: tp.MaybeArray[float],
                       wait: int = 1,
                       flex_2d: bool = True) -> tp.Array2d:

    pos_th = np.asarray(pos_th)
    neg_th = np.asarray(neg_th)
    out = np.full_like(close, 0, dtype=np.float64)

    for col in range(close.shape[1]):
        for i in range(close.shape[0]):
            _pos_th = abs(flex_select_auto_nb(pos_th, i, col, flex_2d))
            _neg_th = abs(flex_select_auto_nb(neg_th, i, col, flex_2d))

            for j in range(i + wait, min(i + window + wait, close.shape[0])):
                if _pos_th > 0 and close[j, col] >= close[i, col] * (1 + _pos_th):
                    out[i, col] = 1
                    break
                if _neg_th > 0 and close[j, col] <= close[i, col] * (1 - _neg_th):
                    out[i, col] = -1
                    break

    return out
```

## 例子

In [ ]:
import numpy as np
from vectorbt.labels.nb import breakout_labels_nb

# 构造示例价格序列（二维数组，shape=(时间, 资产)）
prices = np.array([
    [100],
    [102],
    [101],
    [106],
    [98],
    [110],
    [95]
], dtype=np.float64)

# 设置参数
window = 3      # 未来窗口长度
pos_th = 0.05   # 上涨突破阈值（5%）
neg_th = 0.05   # 下跌突破阈值（5%）
wait = 1        # 等待期（从下一个时刻开始看未来）

# 调用breakout_labels_nb
labels = breakout_labels_nb(
    prices, 
    window=window, 
    pos_th=pos_th, 
    neg_th=neg_th, 
    wait=wait, 
    flex_2d=False  # 单资产时设为False
)

# 展示结果
print("价格序列：", prices.flatten())
print("突破标签：", labels.flatten())
# 1 表示未来窗口内首次上涨突破
# -1 表示未来窗口内首次下跌突破
# 0 表示未来窗口内无突破